# Download from Dataverse 
Our goal is to access ioerDATA via Dataverse API.
In this chapter we'll:
  1. Search the dataset using the DOI
  2. Examine dataset contents (metadata/files)
  3. Download the dataset
  4. Load and visualize data

# Accessing ioerDATA via API

Concept: APIs for Beginners
An API (Application Programming Interface) allows computers to automatically request and obtain data from a data platform. Rather than manually downloading files from a website, the ioerDATA API enables you to access datasets directly within your analysis workflow. This streamlines research, making it more efficient, transparent, and reproducible.

Using ioerDATA as an example:

**Types of Accessible Data**
You can access a range of data, including environmental indicators (such as climate regulation data), urban and regional statistics, spatial datasets (like GeoPackage and shapefiles), and—if permission is granted—restricted socio-economic variables.
**Understanding Metadata and Data**
  - Metadata provides information about the dataset—such as its description, variables, units, source, and license.
  - Data refers to the actual values, numbers, and geometries you will use in your analysis.
**Why Use APIs for Reproducibility?**
  - You can download data automatically with code—no need for manual downloads.
  - You can always retrieve the exact same version of a dataset for future analysis.
  - APIs make your data analysis workflow transparent and easy to repeat.
  - Other researchers can use your code to reproduce your results, supporting transparency and collaboration.

In this step-by-step guide, you'll learn how to search the ioerDATA repository using its API. Instead of manually browsing the website, you'll discover how to find and document datasets efficiently and reproducibly within a Jupyter Notebook.

# Find the PID (Persistent Identifier) via API search
Use this cell to search ioerDATA for the replication package title and automatically extract its PID (often a DOI). This avoids manual copy/paste and prevents “404 Not Found” errors caused by placeholder IDs.

In [4]:
import requests

# ioerDATA base URL (Dataverse instance)
base_url = "https://data.fdz.ioer.de"

# Dataverse search endpoint
search_url = f"{base_url}/api/search"

# Search query (adjust if you want to be more specific)
query = "climate regulation in cities"

params = {
    "q": query,
    "type": "dataset",
    "per_page": 10
}

r = requests.get(search_url, params=params, timeout=30)
r.raise_for_status()

items = r.json().get("data", {}).get("items", [])
if not items:
    raise ValueError(f"No dataset found for query: {query}")

top = items[0]
persistent_id = top.get("global_id")  # PID (often DOI)

print("Top match title:", top.get("name"))
print("Found PID:", persistent_id)

if not persistent_id:
    raise ValueError("Search result did not include 'global_id' (PID). Try refining the search query.")

Top match title: Replication package for: Climate Regulation in Cities
Found PID: doi:10.71830/AFW3N3


This cell searches ioerDATA for the dataset title and stores the dataset’s PID in persistent_id, which we’ll use in the next steps to retrieve metadata and download files reproducibly.

# Fetch full dataset metadata using the PID
Use this cell to download the dataset’s full metadata record from ioerDATA. The metadata includes the file list and flags that indicate whether each file is restricted or open.

In [5]:
import json
import requests

dataset_url = f"{base_url}/api/datasets/:persistentId/"

r = requests.get(dataset_url, params={"persistentId": persistent_id}, timeout=30)
r.raise_for_status()

dataset_metadata = r.json()

# Optional: save metadata locally for documentation/reproducibility
with open("dataset_metadata.json", "w", encoding="utf-8") as f:
    json.dump(dataset_metadata, f, indent=2, ensure_ascii=False)

print("Saved full metadata to: dataset_metadata.json")

# Quick peek: print dataset title from metadata
title = (
    dataset_metadata.get("data", {})
    .get("latestVersion", {})
    .get("metadataBlocks", {})
    .get("citation", {})
    .get("fields", [])
)

print("Metadata retrieved successfully.")

Saved full metadata to: dataset_metadata.json
Metadata retrieved successfully.


This cell uses the PID to fetch the dataset’s full metadata and saves it to dataset_metadata.json. We’ll use the metadata’s file list in the next cell to download only files that are openly accessible.

# Download all openly accessible files into
Use this cell to loop through the dataset’s files and download only those that are openly accessible. We rely on the metadata field restricted to decide what to download (no bypassing restrictions).

In [6]:
import os
import requests

# Where to save files
output_folder = "data/raw"
os.makedirs(output_folder, exist_ok=True)

# Extract file entries from metadata
files = (
    dataset_metadata.get("data", {})
    .get("latestVersion", {})
    .get("files", [])
)

if not files:
    raise ValueError("No files found in metadata. The dataset may have no files or metadata structure differs.")

downloaded = []
skipped = []

for fe in files:
    # Dataverse file structure:
    # fe["restricted"] indicates access restriction at the file level
    # fe["dataFile"] holds file ID, name, size, etc.
    is_restricted = fe.get("restricted", True)
    df = fe.get("dataFile", {})

    file_id = df.get("id")
    filename = df.get("filename", f"file_{file_id}")
    declared_size = df.get("filesize")  # size from metadata (bytes)

    if is_restricted:
        skipped.append((filename, declared_size, "restricted=true in metadata"))
        continue

    if not file_id:
        skipped.append((filename, declared_size, "missing file_id"))
        continue

    # Dataverse datafile download endpoint
    download_url = f"{base_url}/api/access/datafile/{file_id}"

    try:
        # Stream download so large files don’t fill memory
        with requests.get(download_url, stream=True, timeout=60) as r:
            # If file is not actually accessible, Dataverse may return 401/403 here
            r.raise_for_status()

            local_path = os.path.join(output_folder, filename)

            total_bytes = 0
            with open(local_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1MB chunks
                    if chunk:
                        f.write(chunk)
                        total_bytes += len(chunk)

        downloaded.append((filename, total_bytes))

    except requests.exceptions.HTTPError as e:
        skipped.append((filename, declared_size, f"download blocked ({r.status_code})"))
    except requests.exceptions.RequestException as e:
        skipped.append((filename, declared_size, f"network error: {e}"))

# Print results
print("Downloaded files:")
for name, size in downloaded:
    print(f" - {name} ({size:,} bytes)")

print("\nSkipped files:")
for name, size, reason in skipped:
    size_str = f"{size:,} bytes" if isinstance(size, int) else "unknown size"
    print(f" - {name} ({size_str}) -> {reason}")

Downloaded files:
 - Assessment and Monitoring of Local Climate Regulation in Cities by Green Infrastructure—A National Ecosystem Service Indicator for Ge.pdf (11,200,912 bytes)
 - Climate regulation in cities as an ecosystem service_German.pdf (16,359,728 bytes)
 - Cooling_Capacity_2018_buffered.gdb.zip (229,243,224 bytes)
 - Documentation.md (7,247 bytes)
 - Figure_1_Urban_green_Infrastructure_Syrbe_KLu-04.png (471,283 bytes)
 - Map_1_Climate_regulation_Air_Photo.jpg (973,971 bytes)
 - Map_2_Climate_regulation_Tree_Cover.jpg (1,071,537 bytes)
 - Map_3_Climate_regulation_Population.jpg (790,493 bytes)
 - Map_4_Climate_regulation_Cooling_Capacity.jpg (1,145,944 bytes)
 - Map_5_Cities_cooling_capacity.jpg (1,697,718 bytes)
 - Map_6_Cities_inhabitants_cooling_capacity.jpg (1,814,763 bytes)
 - README.md (6,337 bytes)
 - Stadtklima_Skript.py (55,580 bytes)

Skipped files:
 - climate_regulation_in_cities.gpkg (6,545,408 bytes) -> restricted=true in metadata


This cell reads the dataset’s file list from metadata and downloads only files where restricted == False. It saves all downloaded files into data/raw and prints each filename with its size, while clearly listing any skipped files and why they were skipped.

In [2]:
search_url = f"{BASE}/api/search?q={DOI}&type=dataset"
r = requests.get(search_url)
r.raise_for_status()
search_json = r.json()
search_json


{'status': 'OK',
 'data': {'q': '10.71830/AFW3N3',
  'total_count': 34,
  'start': 0,
  'spelling_alternatives': {},
  'items': [{'name': 'Replication package for: Climate Regulation in Cities',
    'type': 'dataset',
    'url': 'https://doi.org/10.71830/AFW3N3',
    'global_id': 'doi:10.71830/AFW3N3',
    'description': 'This dataset presents a national indicator of local climate regulation provided by urban green infrastructure (UGI) across 165 German cities. It quantifies the physical cooling capacity of urban green spaces and the proportion of the population benefiting from these services using geospatial methods.',
    'published_at': '2025-07-22T11:30:49Z',
    'publisher': 'IOER FDZ',
    'citationHtml': 'Syrbe, Ralf-Uwe; Meier, Sophie; Oyshi, Marzan Tasnim, 2025, "Replication package for: Climate Regulation in Cities", <a href="https://doi.org/10.71830/AFW3N3" target="_blank">https://doi.org/10.71830/AFW3N3</a>, ioerDATA, V1',
    'identifier_of_dataverse': 'ioer_fdz',
    'nam

In [3]:
def search_dataset_unique(doi, base=BASE):
    search_url = f"{base}/api/search"
    params = {
        "q": doi,
        "type": "dataset",
        "per_page": 100  # increase page size if many hits
    }
    resp = requests.get(search_url, params=params)
    resp.raise_for_status()
    j = resp.json()
    
    items = j.get("data", {}).get("items", [])
    unique = {}
    for item in items:
        pid = item.get("persistentId")
        if pid is None:
            pid = item.get("id")
        # Only keep the first encountered item per pid
        if pid not in unique:
            unique[pid] = item
    
    # Return deduped list
    return list(unique.values())

# Test it
unique_hits = search_dataset_unique(DOI)
print(f"Found {len(unique_hits)} unique dataset(s):")
for hit in unique_hits:
    print(" •", hit.get("name"), hit.get("persistentId"), "id:", hit.get("id"))

Found 1 unique dataset(s):
 • Replication package for: Climate Regulation in Cities None id: None
